# Note: The building footprints file can be downloaded from the following URLs if needed:
# - https://data.cityofnewyork.us/City-Government/Building-Footprints/5zhs-2jue/about_data
# - https://data.cityofnewyork.us/City-Government/Building-Footprints-Map-/3g6p-4u5s

In [1]:
# Import necessary libraries for data processing and geospatial operations.
import os  # For handling file paths.
import math  # For mathematical operations (not used in this script but imported for potential future use).
import pandas as pd  # For handling tabular data (CSV files).
import geopandas as gpd  # For geospatial data operations (GeoJSON, spatial joins, etc.).
from shapely.geometry import Point  # For creating Point geometries from coordinates.
import shapely  # Import shapely explicitly to access its version.
import tqdm  # Import the tqdm module explicitly to access its version.
# Alias the tqdm function for progress bars to avoid naming conflicts.
from tqdm.auto import tqdm
# Import sys to get the Python version.
import sys

# Print the versions of Python and each imported module.
print(f"Python version: {sys.version}")
print(f"pandas version: {pd.__version__}")
print(f"geopandas version: {gpd.__version__}")
print(f"shapely version: {shapely.__version__}")

# Note: os and math are part of Python's standard library and do not have a separate version.
# Their functionality is tied to the Python version printed above.

Python version: 3.10.12 (main, Nov  6 2024, 20:22:13) [GCC 11.4.0]
pandas version: 2.2.3
geopandas version: 0.14.4
shapely version: 2.0.7


In [2]:
#############################################
# 1. File Paths
#############################################
# Define the base directory where the input datasets are stored.
base_dir = r"/kaggle/input/eyds-base-dataset"

# Define paths to the training and validation CSV files.
train_file = os.path.join(base_dir, "Training_data.csv")
valid_file = os.path.join(base_dir, "Validation_data.csv")

# Define the path to the building footprints GeoJSON file.
buildings_file = os.path.join(base_dir, "Building Footprints_20250222.geojson")

# Note: The building footprints file can be downloaded from the following URLs if needed:
# - https://data.cityofnewyork.us/City-Government/Building-Footprints/5zhs-2jue/about_data
# - https://data.cityofnewyork.us/City-Government/Building-Footprints-Map-/3g6p-4u5s

# Debug: Print the file paths to confirm they are set correctly.
print(f"Debug: Training file path: {train_file}")
print(f"Debug: Validation file path: {valid_file}")
print(f"Debug: Buildings file path: {buildings_file}")

Debug: Training file path: /kaggle/input/eyds-base-dataset/Training_data.csv
Debug: Validation file path: /kaggle/input/eyds-base-dataset/Validation_data.csv
Debug: Buildings file path: /kaggle/input/eyds-base-dataset/Building Footprints_20250222.geojson


In [3]:
#############################################
# 2. Load Datasets
#############################################
# Load the training and validation datasets from CSV files into pandas DataFrames.
# These files must contain 'Latitude' and 'Longitude' columns for geospatial processing.
train_df = pd.read_csv(train_file)
valid_df = pd.read_csv(valid_file)

# Debug: Print the shapes of the loaded DataFrames to confirm they were loaded successfully.
print(f"Debug: Training DataFrame shape: {train_df.shape}")
print(f"Debug: Validation DataFrame shape: {valid_df.shape}")

# Load the building footprints GeoJSON file into a GeoDataFrame.
buildings_gdf = gpd.read_file(buildings_file)

# Debug: Print the columns of the building footprints GeoDataFrame to inspect the data.
print("Debug: Building Footprints Columns:", buildings_gdf.columns.tolist())

# Check if the required 'heightroof' column exists in the building footprints GeoDataFrame.
if "heightroof" not in buildings_gdf.columns:
    raise ValueError("Building footprints file must contain a 'heightroof' column.")

# Convert numeric fields to appropriate types, handling any conversion errors gracefully.
buildings_gdf["heightroof"] = pd.to_numeric(buildings_gdf["heightroof"], errors="coerce")

# If 'shape_area' exists, convert it to numeric type.
if "shape_area" in buildings_gdf.columns:
    buildings_gdf["shape_area"] = pd.to_numeric(buildings_gdf["shape_area"], errors="coerce")

# If 'groundelev' exists, convert it to numeric type.
if "groundelev" in buildings_gdf.columns:
    buildings_gdf["groundelev"] = pd.to_numeric(buildings_gdf["groundelev"], errors="coerce")

# Debug: Print a summary of the building footprints GeoDataFrame after type conversion.
print(f"Debug: Building footprints GeoDataFrame shape: {buildings_gdf.shape}")
print("Debug: Building footprints GeoDataFrame info:")
buildings_gdf.info()

Debug: Training DataFrame shape: (11229, 4)
Debug: Validation DataFrame shape: (1040, 3)
Debug: Building Footprints Columns: ['name', 'base_bbl', 'shape_area', 'heightroof', 'mpluto_bbl', 'cnstrct_yr', 'globalid', 'lststatype', 'feat_code', 'groundelev', 'geomsource', 'bin', 'lstmoddate', 'doitt_id', 'shape_len', 'geometry']
Debug: Building footprints GeoDataFrame shape: (1082823, 16)
Debug: Building footprints GeoDataFrame info:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1082823 entries, 0 to 1082822
Data columns (total 16 columns):
 #   Column      Non-Null Count    Dtype         
---  ------      --------------    -----         
 0   name        2238 non-null     object        
 1   base_bbl    1082823 non-null  object        
 2   shape_area  1082823 non-null  float64       
 3   heightroof  1082822 non-null  float64       
 4   mpluto_bbl  1082823 non-null  object        
 5   cnstrct_yr  1072606 non-null  object        
 6   globalid    1082823 non-null  object    

In [4]:
#############################################
# 3. Convert Points and Buildings to GeoDataFrames in a Metric CRS (EPSG:3857)
#############################################
def create_points_gdf(df):
    """
    Convert a pandas DataFrame with Latitude and Longitude columns into a GeoDataFrame.
    Args:
        df (pd.DataFrame): DataFrame with 'Latitude' and 'Longitude' columns.
    Returns:
        gpd.GeoDataFrame: GeoDataFrame with Point geometries in EPSG:3857 CRS.
    """
    # Create a GeoDataFrame with Point geometries from the Latitude and Longitude columns.
    gdf = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df.Longitude, df.Latitude),
        crs="EPSG:4326"  # Initial CRS is WGS 84 (lat/long).
    )
    # Reproject to EPSG:3857 (Web Mercator) for metric-based distance calculations.
    return gdf.to_crs(epsg=3857)

# Convert the training and validation DataFrames to GeoDataFrames in EPSG:3857.
train_gdf = create_points_gdf(train_df)
valid_gdf = create_points_gdf(valid_df)

# Debug: Print the CRS and shape of the training and validation GeoDataFrames.
print(f"Debug: Training GeoDataFrame CRS: {train_gdf.crs}")
print(f"Debug: Training GeoDataFrame shape: {train_gdf.shape}")
print(f"Debug: Validation GeoDataFrame CRS: {valid_gdf.crs}")
print(f"Debug: Validation GeoDataFrame shape: {valid_gdf.shape}")

# Reproject the building footprints GeoDataFrame to EPSG:3857 for consistent CRS.
buildings_gdf = buildings_gdf.to_crs(epsg=3857)

# Debug: Print the CRS of the building footprints GeoDataFrame after reprojection.
print(f"Debug: Building footprints GeoDataFrame CRS after reprojection: {buildings_gdf.crs}")

# Compute the centroid of each building polygon for spatial queries.
buildings_gdf["centroid"] = buildings_gdf.geometry.centroid

# Build a spatial index for the building footprints GeoDataFrame to optimize spatial queries.
buildings_sindex = buildings_gdf.sindex

# Debug: Confirm that the spatial index was created successfully.
print("Debug: Spatial index created for building footprints GeoDataFrame.")

Debug: Training GeoDataFrame CRS: EPSG:3857
Debug: Training GeoDataFrame shape: (11229, 5)
Debug: Validation GeoDataFrame CRS: EPSG:3857
Debug: Validation GeoDataFrame shape: (1040, 4)
Debug: Building footprints GeoDataFrame CRS after reprojection: EPSG:3857
Debug: Spatial index created for building footprints GeoDataFrame.


In [5]:
#############################################
# 4. Define Thresholds and Optimized Helper Function
#############################################
# Define a list of radius thresholds (in meters) for computing building-based features.
thresholds = [10, 20, 50, 100, 150, 200, 300, 400, 500, 750, 1000]

# Debug: Print the thresholds to confirm they are set correctly.
print(f"Debug: Thresholds for feature computation: {thresholds}")

def enrich_point_features(row, buildings_gdf, sindex, thresholds):
    """
    Compute building-based features for a given point within specified radius thresholds.
    For each radius threshold, calculate:
      - Tallest building height (max heightroof).
      - Average building height.
      - Average and total building footprint area.
    Args:
        row (pd.Series): A row from the GeoDataFrame containing a Point geometry.
        buildings_gdf (gpd.GeoDataFrame): GeoDataFrame containing building footprints.
        sindex (rtree.index.Index): Spatial index for the building footprints GeoDataFrame.
        thresholds (list): List of radius thresholds (in meters).
    Returns:
        pd.Series: Series containing computed features for each threshold.
    """
    point = row.geometry
    features = {}
    
    for r in thresholds:
        # Create a circular buffer of radius r around the point (in EPSG:3857, units are meters).
        buffer_r = point.buffer(r)
        
        # --- Tallest building within the radius ---
        # Use the spatial index to find building footprints that intersect the buffer.
        candidate_idx = list(sindex.query(buffer_r, predicate='intersects'))
        candidates = buildings_gdf.iloc[candidate_idx]
        col_tallest = f"Tallest_Building_{r}m_HEIGHT"
        if not candidates.empty:
            features[col_tallest] = candidates["heightroof"].max()
        else:
            features[col_tallest] = 0
        
        # --- Average building height and area within the radius ---
        # Use the spatial index to find all buildings that intersect the buffer.
        candidate_idx_all = list(sindex.query(buffer_r, predicate='intersects'))
        candidates_all = buildings_gdf.iloc[candidate_idx_all]
        # Filter to only include buildings whose centroids lie within the buffer.
        candidates_all = candidates_all[candidates_all.centroid.within(buffer_r)]
        col_avg_height = f"Average_Building_Height_{r}m"
        col_avg_area = f"Average_Building_Area_{r}m"
        col_total_area = f"Total_Building_Area_{r}m"
        if not candidates_all.empty:
            features[col_avg_height] = candidates_all["heightroof"].mean()
            if "shape_area" in candidates_all.columns:
                features[col_avg_area] = candidates_all["shape_area"].mean()
                features[col_total_area] = candidates_all["shape_area"].sum()
            else:
                features[col_avg_area] = 0
                features[col_total_area] = 0
        else:
            features[col_avg_height] = 0
            features[col_avg_area] = 0
            features[col_total_area] = 0
            
    return pd.Series(features)

# Debug: Print a message to confirm that the helper function is defined.
print("Debug: Helper function 'enrich_point_features' defined successfully.")

Debug: Thresholds for feature computation: [10, 20, 50, 100, 150, 200, 300, 400, 500, 750, 1000]
Debug: Helper function 'enrich_point_features' defined successfully.


In [6]:

train_features = train_gdf.apply(lambda row: enrich_point_features(row, buildings_gdf, buildings_sindex, thresholds), axis=1)
valid_features = valid_gdf.apply(lambda row: enrich_point_features(row, buildings_gdf, buildings_sindex, thresholds), axis=1)

# Merge the computed features into the original GeoDataFrames.
train_enriched = pd.concat([train_gdf, train_features], axis=1)
valid_enriched = pd.concat([valid_gdf, valid_features], axis=1)


In [7]:
# Define the output directory for saving enriched datasets.
sub_dir = r"/kaggle/working/"

# Define the output paths for the enriched training and validation CSV files.
output_train_csv = os.path.join(sub_dir, "Training_data_with_building.csv")
output_valid_csv = os.path.join(sub_dir, "Validation_data_with_building.csv")

# Debug: Print the output file paths to confirm they are set correctly.
print(f"Debug: Output training CSV path: {output_train_csv}")
print(f"Debug: Output validation CSV path: {output_valid_csv}")

Debug: Output training CSV path: /kaggle/working/Training_data_with_building.csv
Debug: Output validation CSV path: /kaggle/working/Validation_data_with_building.csv


In [8]:
#############################################
# 6. Finalize and Save the Enriched Data
#############################################
def finalize_gdf(gdf):
    """
    Finalize a GeoDataFrame by reprojecting to EPSG:4326, extracting Latitude and Longitude, 
    and removing empty and all-zero columns.
    
    Args:
        gdf (gpd.GeoDataFrame): GeoDataFrame to finalize.
    
    Returns:
        pd.DataFrame: DataFrame with Latitude and Longitude columns, without geometry.
    """
    # Reproject the GeoDataFrame back to EPSG:4326 (lat/long).
    gdf = gdf.to_crs(epsg=4326)
    
    # Extract Latitude and Longitude from the geometry column.
    gdf["Latitude"] = gdf.geometry.y
    gdf["Longitude"] = gdf.geometry.x
    
    # Drop the geometry column.
    gdf = gdf.drop(columns="geometry")
    
    # Drop columns that are completely empty (all values are NaN or blank).
    gdf = gdf.dropna(axis=1, how="all")
    
    # Drop columns where all values are zero.
    gdf = gdf.loc[:, (gdf != 0).any(axis=0)]
    
    return gdf

# Finalize the enriched training and validation GeoDataFrames.
train_final = finalize_gdf(train_enriched)
valid_final = finalize_gdf(valid_enriched)

# Debug: Print the shapes of the finalized DataFrames to confirm the transformation.
print(f"Debug: Finalized training DataFrame shape: {train_final.shape}")
print(f"Debug: Finalized validation DataFrame shape: {valid_final.shape}")

# Save the enriched and finalized DataFrames to CSV files.
train_final.to_csv(output_train_csv, index=False)
valid_final.to_csv(output_valid_csv, index=False)

# Debug: Print the final confirmation messages with file paths.
print(f"Debug: Enriched training data saved to: {output_train_csv}")
print(f"Debug: Enriched validation data saved to: {output_valid_csv}")


Debug: Finalized training DataFrame shape: (11229, 26)
Debug: Finalized validation DataFrame shape: (1040, 24)
Debug: Enriched training data saved to: /kaggle/working/Training_data_with_building.csv
Debug: Enriched validation data saved to: /kaggle/working/Validation_data_with_building.csv
